# Citation Link Prediction - Inference Notebook

This notebook performs inference on the test dataset (`test.csv`) using the pre-trained artifacts:
- **Model**: `models/specter2_nn_model.h5` (Keras Deep Neural Network)
- **Scaler**: `models/scaler.pkl` (StandardScaler)
- **Decision Threshold**: `0.35` (optimized via Matthews Correlation Coefficient / MCC)

## 1. Import Dependencies

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from tensorflow.keras.models import load_model

## 2. Configuration & Paths

In [ ]:
DATA_DIR = "data"
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
META_CSV = os.path.join(DATA_DIR, "papers_metadata.csv")
PAPERS_DIR = os.path.join(DATA_DIR, "papers")

MODEL_PATH = "models/specter2_nn_model.h5"
SCALER_PATH = "models/scaler.pkl"
OUTPUT_SUBMISSION = "submission.csv"

# Best threshold from MCC evaluation on validation split
BEST_THRESHOLD = 0.35

## 3. Load Model & Scaler

In [ ]:
print("Loading Scaler...")
scaler = joblib.load(SCALER_PATH)

print("Loading Pre-trained Neural Network Model...")
model = load_model(MODEL_PATH)

print("Model and Scaler loaded successfully!")

## 4. Load Data & Merge Metadata

In [ ]:
test_df = pd.read_csv(TEST_CSV)
meta_df = pd.read_csv(META_CSV)

# Merge metadata for source paper and referenced paper
test_df = test_df.merge(meta_df.add_prefix("paper_"), how="left", left_on="paper", right_on="paper_paper_id")
test_df = test_df.merge(meta_df.add_prefix("ref_"), how="left", left_on="referenced_paper", right_on="ref_paper_id")

for col in ["paper_title", "ref_title"]:
    if col in test_df.columns:
        test_df[col] = test_df[col].fillna("")

print(f"Total test samples: {len(test_df)} rows")
test_df.head()

## 5. Load Document Texts & Extract SPECTER2 Embeddings

In [ ]:
# Get all unique paper_ids in the test set
unique_pids = set(test_df["paper"].dropna().astype(str)).union(set(test_df["referenced_paper"].dropna().astype(str)))

paper_text_map = {}
for pid in tqdm(unique_pids, desc="Loading paper texts"):
    txt_file = os.path.join(PAPERS_DIR, f"{pid}.txt")
    if os.path.exists(txt_file):
        try:
            with open(txt_file, "r", encoding="utf-8", errors="ignore") as f:
                paper_text_map[pid] = f.read().strip()
        except:
            paper_text_map[pid] = ""
    else:
        paper_text_map[pid] = ""

# Load SPECTER2 Model & Tokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained("allenai/specter2_base")
specter_model = AutoModel.from_pretrained("allenai/specter2_base").to(device)
specter_model.eval()

@torch.no_grad()
def encode_specter(texts, batch_size=16):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding Specter2"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=512).to(device)
        output = specter_model(**inputs)
        cls_emb = output.last_hidden_state[:, 0]
        embeddings.append(cls_emb.cpu().numpy())
    return np.vstack(embeddings)

paper_id_list = list(paper_text_map.keys())
paper_texts = [paper_text_map[pid] for pid in paper_id_list]

embeddings_arr = encode_specter(paper_texts)
embed_dict = dict(zip(paper_id_list, embeddings_arr))
print(f"Successfully generated embeddings for {len(embed_dict)} documents.")

## 6. Feature Construction (Textual + Metadata)

In [ ]:
def extract_metadata_features(df):
    feat = []
    for row in df.itertuples():
        p_yr = getattr(row, "paper_publication_year", np.nan)
        r_yr = getattr(row, "ref_publication_year", np.nan)
        yd = p_yr - r_yr if pd.notna(p_yr) and pd.notna(r_yr) else 999
        future = int(yd < 0)
        p_type = getattr(row, "paper_type", None)
        r_type = getattr(row, "ref_type", None)
        same_type = int(p_type == r_type)
        cited_p = getattr(row, "paper_cited_by_count", 0) or 0
        cited_r = getattr(row, "ref_cited_by_count", 0) or 0
        feat.append([yd, future, same_type, cited_p, cited_r])
    return np.array(feat)

def build_features(df):
    X = []
    for row in tqdm(df.itertuples(), total=len(df), desc="Building text features"):
        pid, rid = str(row.paper), str(row.referenced_paper)
        emb1, emb2 = embed_dict.get(pid), embed_dict.get(rid)
        if emb1 is None or emb2 is None:
            # Fallback zero vector if missing
            emb1 = np.zeros(768) if emb1 is None else emb1
            emb2 = np.zeros(768) if emb2 is None else emb2
        concat = np.concatenate([emb1, emb2])
        diff = np.abs(emb1 - emb2)
        features = np.concatenate([concat, diff])
        X.append(features)
    return np.array(X)

X_specter_test = build_features(test_df)
X_meta_test = extract_metadata_features(test_df)
X_test = np.hstack([X_specter_test, X_meta_test])

# Transform using pre-fitted scaler
X_test_scaled = scaler.transform(X_test)
print(f"Final input feature matrix shape: {X_test_scaled.shape}")

## 7. Prediction with Optimal MCC Threshold & Output Submission

In [ ]:
print(f"Predicting probabilities (batch_size=256)...")
y_test_proba = model.predict(X_test_scaled, batch_size=256)

# Apply optimal decision threshold (0.35)
y_test_pred = (y_test_proba >= BEST_THRESHOLD).astype(int).ravel()

# Format submission dataframe
submission = test_df[["id"]].copy()
submission["is_referenced"] = y_test_pred

submission.to_csv(OUTPUT_SUBMISSION, index=False)
print(f"\nSubmission successfully saved to: {OUTPUT_SUBMISSION}")
print("Prediction class distribution:")
print(submission["is_referenced"].value_counts())